# Dois servidores Ollama num notebook do Kaggle (2×T4)

Preparação de referência de dois servidores Ollama, um por placa de vídeo, num notebook do Kaggle
com acelerador **GPU T4 x2** e a opção **Internet** ligada, para `python -m ai_clients generate`
(ver `src/ai_clients/README.md`, Cenário A).

- Ollama fixado na versão **0.35.0** (a da execução final) e modelo `qwen2.5-coder:14b`.
- Cada servidor fica acessível por um túnel rápido do `cloudflared` (endereço `https://...trycloudflare.com`).
- Não grave tokens nem endereços de túnel neste arquivo, e não versione as saídas das células.

## 1. Instalar o Ollama 0.35.0

In [ ]:
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.35.0 sh
!ollama --version

## 2. Subir um servidor por placa (logs em arquivo)

In [ ]:
import os
import subprocess
import time


def serve(port, gpu):
    env = dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{port}", CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"/kaggle/working/ollama-{port}.log", "w")  # noqa: SIM115 - the server keeps it
    return subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)

servers = [serve(11434, 0), serve(11435, 1)]
time.sleep(10)

In [ ]:
!curl -s http://127.0.0.1:11434/api/version; echo
!curl -s http://127.0.0.1:11435/api/version; echo

## 3. Baixar o modelo em cada servidor

In [ ]:
!OLLAMA_HOST=127.0.0.1:11434 ollama pull qwen2.5-coder:14b
!OLLAMA_HOST=127.0.0.1:11435 ollama pull qwen2.5-coder:14b

## 4. Abrir um túnel por servidor

In [ ]:
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /kaggle/working/cloudflared

In [ ]:
import re
from pathlib import Path


def tunnel(port):
    log = open(f"/kaggle/working/tunnel-{port}.log", "w")  # noqa: SIM115 - the tunnel keeps it
    return subprocess.Popen(
        ["/kaggle/working/cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=log, stderr=subprocess.STDOUT,
    )

tunnels = [tunnel(11434), tunnel(11435)]
time.sleep(15)
for name, port in (("GPU0", 11434), ("GPU1", 11435)):
    text = Path(f"/kaggle/working/tunnel-{port}.log").read_text()
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    print(name, found[-1] if found else "(ainda sem endereço: rode esta célula de novo)")

## 5. Usar os endereços

Cole os dois endereços impressos em `GPU0` e `GPU1` do laço em `src/ai_clients/README.md`
(Cenário A, passo 3) e mantenha este notebook rodando enquanto o laço trabalha. O endereço muda toda
vez que o túnel reinicia, e a sessão do Kaggle tem limite de tempo: quando ela cair, rode as células
de novo e troque os endereços no laço.